## 13 — Evento externo protegido ASTA: YOLO × Hough (v3)
Este notebook executa a primeira e única avaliação externa no ASTA. Ele está vinculado
ao contrato imutável publicado pelo Notebook 12 e não permite recalibração por resultado.
Antes de executar qualquer célula, selecione GPU no Colab. Primeiro execute somente
até o gate ASTA_PRONTO_PARA_EVENTO_EXTERNO_UNICO e compartilhe o relatório. A célula de
inferência permanece desativada por padrão.
O processamento é retomável por frame e por método. Cada imagem e máscara é copiada para
o disco local, conferida por SHA-256, processada e removida; o raw no Drive é somente leitura.

In [1]:
%pip install -q ultralytics==8.4.127 opencv-python-headless pillow pandas tqdm scikit-image

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
from PIL import Image, ImageDraw
from skimage.morphology import skeletonize
import gc, hashlib, json, math, os, shutil, time, uuid, zipfile
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics

Image.MAX_IMAGE_PIXELS=200_000_000
drive.mount('/content/drive',force_remount=True)

if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA ausente. Troque o runtime para GPU antes de continuar.')
livre,total=torch.cuda.mem_get_info()
if livre/1024**3<10:
    raise RuntimeError(f'VRAM livre insuficiente: {livre/1024**3:.2f} GB. Reinicie o runtime GPU.')
if ultralytics.__version__!='8.4.127':
    raise RuntimeError(f'Ultralytics divergente: {ultralytics.__version__}')
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True

BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
ASTA_RAW=BACKUP_DIR/'data/raw/asta_real'
AUDIT_FINAL=BACKUP_DIR/'experiments/asta_external_v3/audit/final'
PROTOCOL_FINAL=BACKUP_DIR/'experiments/asta_external_v3/protocol/final'
PROTOCOL_ROOT=BACKUP_DIR/'experiments/asta_external_v3/protocol'
INFERENCE_ROOT=BACKUP_DIR/'experiments/asta_external_v3/inference'
FINAL_MODELS=BACKUP_DIR/'experiments/yolo_final_multiseed_v3/synthetic_v3_c41d5091de56'
HOUGH_CONFIG_PATH=BACKUP_DIR/'data/synthetic_runs/synthetic_v3_c41d5091de56/hough_v3/config_congelada_v3.json'
LOCAL=Path('/content/asta_external_event_v3')
STAGING=LOCAL/'staging_frame'
LOCAL_WEIGHT=LOCAL/'yolo11n_obb_seed_20260824_best.pt'
LOCK_PATH=INFERENCE_ROOT/'TRAVA_EVENTO_EXTERNO_ASTA_V3.json'
for pasta in [LOCAL,STAGING,INFERENCE_ROOT]:pasta.mkdir(parents=True,exist_ok=True)

EXPECTED_PROTOCOL_SHA256='dc4c58381b1e3e7d2a24f7fdac8fc59276e53b9d900853c1191a497319e862bc'
EXPECTED_AUDIT_MANIFEST_SHA256='91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca'
EXPECTED_SKELETON_DIAGNOSTIC_SHA256='cd6a2ccce5a65fee093157c03570259b33f830aa6e6f88286390514cb7bd8388'
EXPECTED_OPERATIONAL_WEIGHT_SHA256='a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c'
EXPECTED_HOUGH_CONFIG_SHA256='14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5'
EXPECTED_FINAL_MODELS_CONTRACT_SHA256='0a8d3537969dcb033dc8cdfa1a9d63dc24de4cc98e212c42fab62621126890a7'
BATCH_TILES_YOLO=16

def agora():return datetime.now().strftime('%H:%M:%S')
def sha256_arquivo(caminho,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for bloco in iter(lambda:arq.read(chunk),b''):h.update(bloco)
    return h.hexdigest()
def sha256_canonico(objeto):
    texto=json.dumps(objeto,sort_keys=True,separators=(',',':'),ensure_ascii=False)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()
def verificar_hash_canonico(registro,campo,ensure_ascii=False):
    copia=dict(registro);observado=copia.pop(campo)
    texto=json.dumps(copia,sort_keys=True,separators=(',',':'),ensure_ascii=ensure_ascii)
    calculado=hashlib.sha256(texto.encode('utf-8')).hexdigest()
    if calculado!=observado:raise RuntimeError(f'Hash canônico divergente: {campo}')
    return observado
def gravar_json_atomico(caminho,objeto):
    caminho=Path(caminho);caminho.parent.mkdir(parents=True,exist_ok=True)
    temporario=caminho.with_name(caminho.name+'.tmp')
    temporario.write_text(json.dumps(objeto,indent=2,ensure_ascii=False),encoding='utf-8')
    os.replace(temporario,caminho)
def gravar_csv_atomico(caminho,tabela):
    caminho=Path(caminho);caminho.parent.mkdir(parents=True,exist_ok=True)
    temporario=caminho.with_name(caminho.name+'.tmp');tabela.to_csv(temporario,index=False)
    os.replace(temporario,caminho)

print(json.dumps({
    'status':'GPU_ASTA_APROVADA','gpu':torch.cuda.get_device_name(0),
    'vram_total_gb':round(total/1024**3,2),'vram_livre_gb':round(livre/1024**3,2),
    'torch':torch.__version__,'cuda':torch.version.cuda,'ultralytics':ultralytics.__version__
},indent=2,ensure_ascii=False))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Mounted at /content/drive
{
  "status": "GPU_ASTA_APROVADA",
  "gpu": "Tesla T4",
  "vram_total_gb": 14.56,
  "vram_livre_gb": 14.46,
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "ultralytics": "8.4.127"
}


## 1. Gate pré-evento — sem inferência
Este gate valida o contrato publicado, o inventário, o checkpoint operacional e o Hough
congelado. Os arquivos grandes não são copiados aqui: cada par será conferido por hash no
momento em que for processado. Compartilhe o JSON antes de habilitar a seção 3.

In [2]:
contrato_path=PROTOCOL_FINAL/'contrato_protocolo_asta_v3.json'
ponteiro_path=PROTOCOL_ROOT/'LATEST_ASTA_PROTOCOL_V3.json'
manifest_path=AUDIT_FINAL/'manifest_auditoria_asta_v3.csv'
diagnostico_path=PROTOCOL_FINAL/'diagnostico_skeleton_asta_v3.csv'
assert contrato_path.is_file() and ponteiro_path.is_file()

contrato=json.loads(contrato_path.read_text(encoding='utf-8'))
ponteiro=json.loads(ponteiro_path.read_text(encoding='utf-8'))
assert verificar_hash_canonico(contrato,'sha256_contrato_protocolo_asta')==EXPECTED_PROTOCOL_SHA256
assert ponteiro['sha256_contrato_protocolo_asta']==EXPECTED_PROTOCOL_SHA256
assert contrato['status']=='APROVADO_SEM_INFERENCIA' and contrato['aprovado'] is True
assert contrato['raw_somente_leitura'] is True
assert contrato['yolo_executado'] is False and contrato['hough_executado'] is False
assert contrato['inferencia_asta_executada'] is False
cfg=contrato['configuracao']
assert cfg['sha256_protocolo_candidato']=='33afc29c4479c0129bad8f4854f373215d588bc2566d54404e5770fc94bc02c1'
assert cfg['tiles']=={'tamanho':512,'overlap':128,'stride':384,'grid_x':28,'grid_y':28,
    'tiles_por_frame':784,'tiles_total':139552,'padding_direita':320,'padding_abaixo':320,
    'grid_regular':True,'descartar_predicao_centro_fora_frame':True}
assert cfg['modelo_yolo']['seed_operacional']==20260824
assert cfg['modelo_yolo']['threshold_confianca']==0.45
assert cfg['metricas']['primaria']=='f1_centerline_macro_frames_positivos'
assert cfg['metricas']['tolerancia_centerline_px']==14.0
assert cfg['restricoes']['ajuste_por_resultado_asta'] is False

assert sha256_arquivo(manifest_path)==EXPECTED_AUDIT_MANIFEST_SHA256
assert sha256_arquivo(diagnostico_path)==EXPECTED_SKELETON_DIAGNOSTIC_SHA256
manifest=pd.read_csv(manifest_path).sort_values('id_asta').reset_index(drop=True)
assert len(manifest)==178 and manifest.id_asta.nunique()==178
assert int((manifest.mascara_pixels_positivos>0).sum())==177
assert manifest.loc[manifest.mascara_pixels_positivos==0,'id_asta'].tolist()==['ML1_20200328_002539_red']
for linha in manifest.itertuples():
    if not (ASTA_RAW/linha.arquivo_imagem).is_file():raise FileNotFoundError(linha.arquivo_imagem)
    if not (ASTA_RAW/linha.arquivo_mascara).is_file():raise FileNotFoundError(linha.arquivo_mascara)

contrato_modelos=json.loads((FINAL_MODELS/'contrato_modelos_finais_v3.json').read_text(encoding='utf-8'))
assert verificar_hash_canonico(contrato_modelos,'sha256_contrato_modelos_finais',ensure_ascii=True)==EXPECTED_FINAL_MODELS_CONTRACT_SHA256
operacional=next(x for x in contrato_modelos['checkpoints'] if int(x['seed'])==20260824)
peso_operacional=BACKUP_DIR/operacional['best_pt_relativo_drive']
assert peso_operacional.is_file() and sha256_arquivo(peso_operacional)==EXPECTED_OPERATIONAL_WEIGHT_SHA256
hough_config=json.loads(HOUGH_CONFIG_PATH.read_text(encoding='utf-8'))
assert hough_config['sha256_config']==EXPECTED_HOUGH_CONFIG_SHA256

CONFIG_EVENTO={
    'protocolo':'asta_external_protected_event_v3','versao':'3.0',
    'sha256_contrato_protocolo_asta':EXPECTED_PROTOCOL_SHA256,
    'sha256_manifest_auditoria':EXPECTED_AUDIT_MANIFEST_SHA256,
    'sha256_checkpoint_yolo':EXPECTED_OPERATIONAL_WEIGHT_SHA256,
    'sha256_hough_config':EXPECTED_HOUGH_CONFIG_SHA256,
    'frames':178,'positivos':177,'negativos':1,'tiles_total':139552,
    'batch_tiles_yolo':BATCH_TILES_YOLO,'dispositivo_yolo':'cuda:0','precisao_yolo':'fp32',
    'determinismo':{'cudnn_benchmark':False,'cudnn_deterministic':True,
        'hough_rng':'seed_sha256_por_frame_tile'},
    'checkpoint':'por_frame_e_metodo','retomada':True,
    'timing':{'preprocessamento':'decode_e_normalizacao_da_imagem_mais_extracao_dos_tiles',
        'inferencia':'chamada_do_detector_e_conversao_local_das_saidas_com_sincronizacao_cuda',
        'merging':'uniao_global_de_segmentos',
        'total_frame':'preprocessamento_mais_inferencia_mais_merging',
        'exclui':['copia_drive','decode_mascara','skeleton_ground_truth','avaliacao']},
    'resultado_primario':'media_macro_f1_centerline_nos_177_frames_positivos',
    'comparacao':'delta_pareado_yolo_menos_hough_por_frame',
    'bootstrap_replicacoes':2000,'bootstrap_seed':20260823,
    'ajuste_pos_resultado':False
}
CONFIG_EVENTO['sha256_config_evento']=sha256_canonico(CONFIG_EVENTO)
gravar_json_atomico(LOCAL/'config_evento_asta_v3.json',CONFIG_EVENTO)
evento_existente=json.loads(LOCK_PATH.read_text(encoding='utf-8')) if LOCK_PATH.is_file() else None
gate={
    'status':'ASTA_PRONTO_PARA_EVENTO_EXTERNO_UNICO',
    'sha256_contrato_protocolo_asta':EXPECTED_PROTOCOL_SHA256,
    'sha256_config_evento':CONFIG_EVENTO['sha256_config_evento'],
    'pares':178,'tiles_total':139552,'checkpoint_operacional_seed':20260824,
    'sha256_checkpoint_operacional':EXPECTED_OPERATIONAL_WEIGHT_SHA256,
    'sha256_hough_config':EXPECTED_HOUGH_CONFIG_SHA256,
    'gpu':torch.cuda.get_device_name(0),'batch_tiles_yolo':BATCH_TILES_YOLO,
    'evento_existente':None if evento_existente is None else {
        'event_id':evento_existente.get('event_id'),'status':evento_existente.get('status'),
        'sha256_config_evento':evento_existente.get('sha256_config_evento')},
    'inferencia_executada_nesta_sessao':False
}
print(json.dumps(gate,indent=2,ensure_ascii=False))
print('Gate pré-evento aprovado. Não habilite a inferência antes de compartilhar este relatório.')

{
  "status": "ASTA_PRONTO_PARA_EVENTO_EXTERNO_UNICO",
  "sha256_contrato_protocolo_asta": "dc4c58381b1e3e7d2a24f7fdac8fc59276e53b9d900853c1191a497319e862bc",
  "sha256_config_evento": "81ab0148b765fd3664b185812560826b21bf680081aa6c17d37ee2b94f773137",
  "pares": 178,
  "tiles_total": 139552,
  "checkpoint_operacional_seed": 20260824,
  "sha256_checkpoint_operacional": "a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c",
  "sha256_hough_config": "14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5",
  "gpu": "Tesla T4",
  "batch_tiles_yolo": 16,
  "evento_existente": null,
  "inferencia_executada_nesta_sessao": false
}
Gate pré-evento aprovado. Não habilite a inferência antes de compartilhar este relatório.


## 2. Funções congeladas e testes determinísticos
A máscara oficial continua semântica. O matching compara pixels das centerlines com
tolerância simétrica de 14 px. Fragmentos de tiles só são unidos quando satisfazem
simultaneamente ângulo, distância perpendicular e gap axial do contrato.

In [3]:
TILE=cfg['tiles']['tamanho'];STRIDE=cfg['tiles']['stride']
STARTS_X=[i*STRIDE for i in range(cfg['tiles']['grid_x'])]
STARTS_Y=[i*STRIDE for i in range(cfg['tiles']['grid_y'])]

def copiar_com_hash(origem,destino,esperado,tentativas=3,chunk=8*1024*1024):
    ultimo=None
    for tentativa in range(1,tentativas+1):
        try:
            if destino.exists():destino.unlink()
            h=hashlib.sha256()
            with origem.open('rb') as entrada,destino.open('wb') as saida:
                for bloco in iter(lambda:entrada.read(chunk),b''):
                    saida.write(bloco);h.update(bloco)
            observado=h.hexdigest()
            if observado!=esperado:raise IOError(f'hash divergente: {observado}')
            return observado
        except (OSError,IOError) as exc:
            ultimo=exc
            if destino.exists():destino.unlink()
            if tentativa<tentativas:time.sleep(2**tentativa)
    raise IOError(f'Falha após {tentativas} tentativas: {ultimo}')

def percentil_histograma_uint8(imagem,percentil):
    hist=np.bincount(imagem.reshape(-1),minlength=256)
    alvo=(imagem.size-1)*(float(percentil)/100.0)
    return int(np.searchsorted(np.cumsum(hist),alvo,side='right'))

def normalizar_frame_uint8(imagem):
    baixo=percentil_histograma_uint8(imagem,cfg['preprocessamento']['percentil_baixo'])
    alto=percentil_histograma_uint8(imagem,cfg['preprocessamento']['percentil_alto'])
    if alto<=baixo:alto=min(255,baixo+1)
    valores=np.arange(256,dtype=np.float32)
    lut=np.clip((valores-baixo)*255.0/(alto-baixo),0,255).round().astype(np.uint8)
    normalizada=lut[imagem]
    mediana=percentil_histograma_uint8(normalizada,50.0)
    return normalizada,{'percentil_baixo_valor':baixo,'percentil_alto_valor':alto,'mediana_normalizada':mediana}

def extrair_tile(imagem,x,y,preenchimento):
    h,w=imagem.shape;tile=np.full((TILE,TILE),preenchimento,dtype=np.uint8)
    valido_h=max(0,min(TILE,h-y));valido_w=max(0,min(TILE,w-x))
    if valido_h and valido_w:tile[:valido_h,:valido_w]=imagem[y:y+valido_h,x:x+valido_w]
    return tile

def caixa_para_eixo_maior(pontos):
    pts=np.asarray(pontos,dtype=float).reshape(4,2);centro=pts.mean(axis=0)
    cov=(pts-centro).T@(pts-centro)
    valores,vetores=np.linalg.eigh(cov);direcao=vetores[:,int(np.argmax(valores))]
    proj=(pts-centro)@direcao;menor,maior=float(proj.min()),float(proj.max())
    p1=centro+direcao*menor;p2=centro+direcao*maior
    return p1,p2

def angulo_segmento(seg):
    dx=seg['x2']-seg['x1'];dy=seg['y2']-seg['y1']
    return math.degrees(math.atan2(dy,dx))%180.0

def diferenca_angular(a,b):
    d=abs(a-b)%180.0
    return min(d,180.0-d)

def segmentos_compativeis(a,b):
    limite=cfg['merging_global']
    aa,ab=angulo_segmento(a),angulo_segmento(b)
    if diferenca_angular(aa,ab)>limite['angulo_max_graus']:return False
    va=np.array([a['x2']-a['x1'],a['y2']-a['y1']],float)
    vb=np.array([b['x2']-b['x1'],b['y2']-b['y1']],float)
    va/=max(np.linalg.norm(va),1e-9);vb/=max(np.linalg.norm(vb),1e-9)
    if np.dot(va,vb)<0:vb=-vb
    eixo=va+vb;eixo/=max(np.linalg.norm(eixo),1e-9);normal=np.array([-eixo[1],eixo[0]])
    ma=np.array([(a['x1']+a['x2'])/2,(a['y1']+a['y2'])/2])
    mb=np.array([(b['x1']+b['x2'])/2,(b['y1']+b['y2'])/2])
    if abs(float((mb-ma)@normal))>limite['distancia_perpendicular_max_px']:return False
    pa=sorted([float(np.array([a['x1'],a['y1']])@eixo),float(np.array([a['x2'],a['y2']])@eixo)])
    pb=sorted([float(np.array([b['x1'],b['y1']])@eixo),float(np.array([b['x2'],b['y2']])@eixo)])
    gap=max(0.0,max(pa[0],pb[0])-min(pa[1],pb[1]))
    return gap<=limite['gap_axial_max_px']

def agregar_componente(segmentos,indices):
    pontos=[];pesos=[]
    for indice in indices:
        s=segmentos[indice];peso=max(float(s['peso']),1e-6)
        pontos.extend([[s['x1'],s['y1']],[s['x2'],s['y2']]]);pesos.extend([peso,peso])
    pts=np.asarray(pontos,float);w=np.asarray(pesos,float);centro=np.average(pts,axis=0,weights=w)
    desloc=pts-centro;cov=(desloc*w[:,None]).T@desloc/max(w.sum(),1e-9)
    valores,vetores=np.linalg.eigh(cov);eixo=vetores[:,int(np.argmax(valores))]
    proj=desloc@eixo;p1=centro+eixo*proj.min();p2=centro+eixo*proj.max()
    confiancas=[float(segmentos[i].get('confianca',0.0)) for i in indices]
    return {'x1':float(p1[0]),'y1':float(p1[1]),'x2':float(p2[0]),'y2':float(p2[1]),
            'comprimento':float(np.linalg.norm(p2-p1)),'confianca_max':max(confiancas,default=0.0),
            'fragmentos':len(indices)}

def unir_segmentos(segmentos):
    n=len(segmentos)
    if n==0:return []
    pai=list(range(n))
    def raiz(i):
        while pai[i]!=i:pai[i]=pai[pai[i]];i=pai[i]
        return i
    def unir(i,j):
        ri,rj=raiz(i),raiz(j)
        if ri!=rj:pai[rj]=ri
    buckets={}
    for i,s in enumerate(segmentos):buckets.setdefault((int(s['tile_x']),int(s['tile_y'])),[]).append(i)
    for (tx,ty),indices in buckets.items():
        for ny in range(ty-1,ty+2):
            for nx in range(tx-1,tx+2):
                for i in indices:
                    for j in buckets.get((nx,ny),[]):
                        if j<=i:continue
                        if segmentos_compativeis(segmentos[i],segmentos[j]):unir(i,j)
    grupos={}
    for i in range(n):grupos.setdefault(raiz(i),[]).append(i)
    return [agregar_componente(segmentos,indices) for indices in grupos.values()]

def rasterizar_segmentos(segmentos,shape):
    h,w=shape;canvas=np.zeros((h,w),np.uint8)
    for s in segmentos:
        p1=(int(round(s['x1'])),int(round(s['y1'])));p2=(int(round(s['x2'])),int(round(s['y2'])))
        ok,q1,q2=cv2.clipLine((0,0,w,h),p1,p2)
        if ok:cv2.line(canvas,q1,q2,1,1,cv2.LINE_8)
    return canvas

def metricas_centerline(segmentos,skeleton_gt,tolerancia=14):
    pred=rasterizar_segmentos(segmentos,skeleton_gt.shape);gt=skeleton_gt.astype(np.uint8)
    kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*tolerancia+1,2*tolerancia+1))
    gt_dilatado=cv2.dilate(gt,kernel);pred_dilatado=cv2.dilate(pred,kernel)
    pred_pixels=int(pred.sum());gt_pixels=int(gt.sum())
    pred_hits=int(((pred>0)&(gt_dilatado>0)).sum())
    gt_hits=int(((gt>0)&(pred_dilatado>0)).sum())
    precisao=(pred_hits/pred_pixels) if pred_pixels else (1.0 if gt_pixels==0 else 0.0)
    recall=(gt_hits/gt_pixels) if gt_pixels else None
    f1=(2*precisao*recall/(precisao+recall)) if recall is not None and (precisao+recall)>0 else (0.0 if gt_pixels else None)
    return {'pred_pixels':pred_pixels,'gt_pixels':gt_pixels,'pred_hits':pred_hits,'gt_hits':gt_hits,
            'precisao_centerline':float(precisao),'recall_centerline':None if recall is None else float(recall),
            'f1_centerline':None if f1 is None else float(f1),'cobertura_gt':None if recall is None else float(recall)}

# Regressões locais: grid, normalização, merging e tolerância centerline.
assert len(STARTS_X)==len(STARTS_Y)==28 and STARTS_X[-1]+TILE==10880
teste=np.arange(256,dtype=np.uint8).reshape(16,16);norm,meta=normalizar_frame_uint8(teste)
assert norm.dtype==np.uint8 and norm.min()==0 and norm.max()==255 and meta['percentil_alto_valor']>meta['percentil_baixo_valor']
base={'x1':10.,'y1':20.,'x2':300.,'y2':20.,'peso':1.,'confianca':0.9,'tile_x':0,'tile_y':0}
duplicado={'x1':280.,'y1':21.,'x2':560.,'y2':21.,'peso':1.,'confianca':0.8,'tile_x':1,'tile_y':0}
paralelo={'x1':10.,'y1':50.,'x2':300.,'y2':50.,'peso':1.,'confianca':0.7,'tile_x':0,'tile_y':0}
assert len(unir_segmentos([base,duplicado]))==1 and len(unir_segmentos([base,paralelo]))==2
gt=np.zeros((128,128),bool);gt[64,10:110]=True
identico=metricas_centerline([{'x1':10,'y1':64,'x2':109,'y2':64}],gt,14)
distante=metricas_centerline([{'x1':10,'y1':90,'x2':109,'y2':90}],gt,14)
assert identico['f1_centerline']==1.0 and distante['f1_centerline']==0.0
print('Testes determinísticos do protocolo externo aprovados; nenhuma inferência executada.')

Testes determinísticos do protocolo externo aprovados; nenhuma inferência executada.


## 3. Smoke operacional sem dados ASTA
Este smoke carrega o checkpoint operacional e executa dois tiles artificiais. Ele valida
lote, GPU, OBB e Hough sem ler nenhuma imagem ou máscara ASTA e sem criar a trava do evento.

In [4]:
print(f'[{agora()}] Smoke operacional sem dados ASTA: início')
copiar_com_hash(peso_operacional,LOCAL_WEIGHT,EXPECTED_OPERATIONAL_WEIGHT_SHA256)
from ultralytics import YOLO
modelo_smoke=YOLO(str(LOCAL_WEIGHT))
tile_vazio=np.zeros((512,512),np.uint8)
tile_linha=np.zeros((512,512),np.uint8);cv2.line(tile_linha,(24,420),(488,70),220,3,cv2.LINE_AA)
entradas_smoke=[cv2.cvtColor(x,cv2.COLOR_GRAY2BGR) for x in [tile_vazio,tile_linha]]
torch.cuda.reset_peak_memory_stats();torch.cuda.synchronize();inicio_smoke=time.perf_counter()
resultados_smoke=modelo_smoke.predict(source=entradas_smoke,imgsz=512,conf=0.45,iou=0.7,
    max_det=100,device=0,verbose=False,batch=2,half=False)
torch.cuda.synchronize();tempo_smoke=time.perf_counter()-inicio_smoke
assert len(resultados_smoke)==2
caixas_smoke=[0 if r.obb is None else int(len(r.obb)) for r in resultados_smoke]
d_smoke=hough_config['detector'];bordas_smoke=cv2.Canny(tile_linha,int(d_smoke['canny_lo']),int(d_smoke['canny_hi']))
cv2.setRNGSeed(20260823)
linhas_smoke=cv2.HoughLinesP(bordas_smoke,1,np.pi/180,threshold=int(d_smoke['hough_threshold']),
    minLineLength=max(1,int(round(512*float(d_smoke['min_length_frac'])))),maxLineGap=int(d_smoke['max_gap']))
assert linhas_smoke is not None and np.asarray(linhas_smoke).reshape(-1,4).shape[0]>0
pico_vram=torch.cuda.max_memory_allocated()/1024**3
relatorio_smoke={'status':'SMOKE_IMPLEMENTACAO_EVENTO_ASTA_APROVADO_SEM_DADOS_ASTA',
    'sha256_config_evento':CONFIG_EVENTO['sha256_config_evento'],'tiles_artificiais':2,
    'resultados_yolo':len(resultados_smoke),'obb_por_tile':caixas_smoke,
    'linhas_hough_tile_artificial':int(np.asarray(linhas_smoke).reshape(-1,4).shape[0]),
    'tempo_yolo_lote_segundos':round(tempo_smoke,4),'pico_vram_gb':round(pico_vram,3),
    'asta_lido':False,'trava_evento_criada':False}
print(json.dumps(relatorio_smoke,indent=2,ensure_ascii=False))
del modelo_smoke,resultados_smoke,entradas_smoke,tile_vazio,tile_linha,bordas_smoke,linhas_smoke
gc.collect();torch.cuda.empty_cache()

[14:59:24] Smoke operacional sem dados ASTA: início
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
{
  "status": "SMOKE_IMPLEMENTACAO_EVENTO_ASTA_APROVADO_SEM_DADOS_ASTA",
  "sha256_config_evento": "81ab0148b765fd3664b185812560826b21bf680081aa6c17d37ee2b94f773137",
  "tiles_artificiais": 2,
  "resultados_yolo": 2,
  "obb_por_tile": [
    0,
    1
  ],
  "linhas_hough_tile_artificial": 3,
  "tempo_yolo_lote_segundos": 10.1581,
  "pico_vram_gb": 0.071,
  "asta_lido": false,
  "trava_evento_criada": false
}


## 4. Evento único, retomável por frame e por método
Após a aprovação do gate pré-evento, altere apenas EXECUTAR_EVENTO_ASTA=True.
Em uma sessão retomada, use também RETOMAR_EVENTO_INCOMPLETO=True. Nunca use a opção
de retomada para um evento já concluído.
O progresso possui uma barra externa de 178 frames e barras internas de 784 tiles. O
checkpoint de cada método é publicado atomicamente logo após o frame ser concluído.

In [6]:
import json
from pathlib import Path

trava = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
event_id = trava["event_id"]
event_dir = INFERENCE_ROOT / f"evento_{event_id}"

resultado = {
    "event_id": event_id,
    "status_trava": trava.get("status"),
    "sha256_config_evento": trava.get("sha256_config_evento"),
    "yolo_checkpoints": len(list((event_dir / "checkpoints/yolo").glob("*.json"))),
    "hough_checkpoints": len(list((event_dir / "checkpoints/hough").glob("*.json"))),
    "arquivos_temporarios": [
        str(p.relative_to(event_dir))
        for p in event_dir.rglob("*.tmp")
    ],
    "staging_local": [
        {"arquivo": p.name, "tamanho_mb": round(p.stat().st_size / 1024**2, 2)}
        for p in STAGING.glob("*")
        if p.is_file()
    ],
}

print(json.dumps(resultado, indent=2, ensure_ascii=False))

{
  "event_id": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "status_trava": "em_andamento",
  "sha256_config_evento": "81ab0148b765fd3664b185812560826b21bf680081aa6c17d37ee2b94f773137",
  "yolo_checkpoints": 1,
  "hough_checkpoints": 0,
  "arquivos_temporarios": [],
  "staging_local": [
    {
      "arquivo": "ML1_20170902_022840_red_mask.png",
      "tamanho_mb": 0.15
    },
    {
      "arquivo": "ML1_20170902_022840_red.fits_full.png",
      "tamanho_mb": 96.11
    }
  ]
}


In [7]:
import time
import json
import numpy as np
import cv2
from tqdm.auto import tqdm

imagens_staging = [
    p for p in STAGING.glob("*")
    if p.is_file() and not p.name.endswith("_mask.png")
]

assert len(imagens_staging) == 1, imagens_staging
img_local = imagens_staging[0]

t0 = time.perf_counter()
imagem = cv2.imread(str(img_local), cv2.IMREAD_GRAYSCALE)
assert imagem is not None and imagem.shape == (10560, 10560)

imagem_norm, meta_norm = normalizar_frame_uint8(imagem)
tempo_normalizacao = time.perf_counter() - t0
preenchimento = int(meta_norm["mediana_normalizada"])

coordenadas = [
    (ix, iy, x, y)
    for iy, y in enumerate(STARTS_Y)
    for ix, x in enumerate(STARTS_X)
]

# Amostra espacialmente distribuída de 12 dos 784 tiles.
indices = np.linspace(
    0, len(coordenadas) - 1, 12, dtype=int
).tolist()

d = hough_config["detector"]
tempos_hough = []
linhas_por_tile = []
segmentos_amostra = []

for indice in tqdm(indices, desc="Perfil Hough", unit="tile"):
    ix, iy, x, y = coordenadas[indice]
    tile = extrair_tile(imagem_norm, x, y, preenchimento)

    base = tile
    if float(d.get("pre_blur_sigma", 0.0)) > 0:
        base = cv2.GaussianBlur(
            tile, (0, 0), float(d["pre_blur_sigma"])
        )

    bordas = cv2.Canny(
        base,
        int(d["canny_lo"]),
        int(d["canny_hi"]),
    )

    seed_hough = (
        int(
            hashlib.sha256(
                f"{img_local.stem}|{ix}|{iy}|hough_v3".encode()
            ).hexdigest()[:8],
            16,
        )
        & 0x7fffffff
    )
    cv2.setRNGSeed(seed_hough)

    inicio = time.perf_counter()
    linhas = cv2.HoughLinesP(
        bordas,
        1,
        np.pi / 180,
        threshold=int(d["hough_threshold"]),
        minLineLength=max(
            1,
            int(round(TILE * float(d["min_length_frac"]))),
        ),
        maxLineGap=int(d["max_gap"]),
    )
    tempos_hough.append(time.perf_counter() - inicio)

    linhas_array = (
        np.empty((0, 4), dtype=int)
        if linhas is None
        else np.asarray(linhas).reshape(-1, 4)
    )
    linhas_por_tile.append(len(linhas_array))

    for x1, y1, x2, y2 in linhas_array:
        p1 = np.array([float(x1 + x), float(y1 + y)])
        p2 = np.array([float(x2 + x), float(y2 + y)])
        comprimento = float(np.linalg.norm(p2 - p1))
        segmentos_amostra.append({
            "x1": float(p1[0]),
            "y1": float(p1[1]),
            "x2": float(p2[0]),
            "y2": float(p2[1]),
            "peso": comprimento,
            "confianca": 0.0,
            "comprimento": comprimento,
            "tile_x": ix,
            "tile_y": iy,
        })

tempo_merge = None
quantidade_unida = None
merge_pulado = False

# Evita criar outro processamento demorado durante o diagnóstico.
if len(segmentos_amostra) <= 3000:
    inicio = time.perf_counter()
    unidos = unir_segmentos(segmentos_amostra)
    tempo_merge = time.perf_counter() - inicio
    quantidade_unida = len(unidos)
else:
    merge_pulado = True

relatorio = {
    "evento_diagnosticado": event_id,
    "frame": img_local.stem,
    "tiles_amostrados": len(indices),
    "tempo_normalizacao_frame_s": round(tempo_normalizacao, 4),
    "hough_s_por_tile": {
        "min": round(float(np.min(tempos_hough)), 6),
        "mediana": round(float(np.median(tempos_hough)), 6),
        "media": round(float(np.mean(tempos_hough)), 6),
        "max": round(float(np.max(tempos_hough)), 6),
    },
    "projecao_hough_784_tiles_s": round(
        float(np.mean(tempos_hough) * 784), 2
    ),
    "linhas_por_tile": {
        "min": int(np.min(linhas_por_tile)),
        "mediana": float(np.median(linhas_por_tile)),
        "media": round(float(np.mean(linhas_por_tile)), 2),
        "max": int(np.max(linhas_por_tile)),
        "total_amostra": int(np.sum(linhas_por_tile)),
    },
    "segmentos_amostra": len(segmentos_amostra),
    "tempo_merge_amostra_s": (
        None if tempo_merge is None else round(tempo_merge, 4)
    ),
    "segmentos_apos_merge": quantidade_unida,
    "merge_pulado_por_volume": merge_pulado,
    "checkpoint_yolo_preservado": 1,
    "checkpoint_hough_preservado": 0,
}

print(json.dumps(relatorio, indent=2, ensure_ascii=False))

Perfil Hough:   0%|          | 0/12 [00:00<?, ?tile/s]

{
  "evento_diagnosticado": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "frame": "ML1_20170902_022840_red.fits_full",
  "tiles_amostrados": 12,
  "tempo_normalizacao_frame_s": 4.1624,
  "hough_s_por_tile": {
    "min": 0.007827,
    "mediana": 0.097521,
    "media": 0.091237,
    "max": 0.108337
  },
  "projecao_hough_784_tiles_s": 71.53,
  "linhas_por_tile": {
    "min": 10,
    "mediana": 396.5,
    "media": 365.42,
    "max": 428,
    "total_amostra": 4385
  },
  "segmentos_amostra": 4385,
  "tempo_merge_amostra_s": null,
  "segmentos_apos_merge": null,
  "merge_pulado_por_volume": true,
  "checkpoint_yolo_preservado": 1,
  "checkpoint_hough_preservado": 0
}


In [8]:
import json

checkpoint_yolo = next(
    (event_dir / "checkpoints/yolo").glob("*.json")
)
registro_yolo = json.loads(
    checkpoint_yolo.read_text(encoding="utf-8")
)

relatorio_tempo_yolo = {
    "id_asta": registro_yolo["id_asta"],
    "segmentos_brutos": registro_yolo["segmentos_brutos"],
    "segmentos_apos_merging": registro_yolo["segmentos_apos_merging"],
    "tempos_segundos": registro_yolo["tempos_segundos"],
}

print(json.dumps(
    relatorio_tempo_yolo,
    indent=2,
    ensure_ascii=False,
))

{
  "id_asta": "ML1_20170902_022840_red",
  "segmentos_brutos": 25,
  "segmentos_apos_merging": 18,
  "tempos_segundos": {
    "copia_drive": 2.484507121999968,
    "preprocessamento": 3.957024380000348,
    "inferencia": 3.3404905800005054,
    "merging": 0.005654710000044361,
    "avaliacao": 2.105555532999915,
    "total_frame": 7.303169670000898
  }
}


In [9]:
# Preserva a implementação original apenas para o teste de equivalência.
unir_segmentos_escalar_validacao = unir_segmentos

def unir_segmentos(segmentos):
    """Mesmo grafo do predicado escalar, com pares avaliados em blocos NumPy."""
    n = len(segmentos)
    if n == 0:
        return []

    pai = list(range(n))

    def raiz(i):
        while pai[i] != i:
            pai[i] = pai[pai[i]]
            i = pai[i]
        return i

    def unir(i, j):
        ri, rj = raiz(i), raiz(j)
        if ri != rj:
            pai[rj] = ri

    p1 = np.asarray(
        [[s["x1"], s["y1"]] for s in segmentos],
        dtype=np.float64,
    )
    p2 = np.asarray(
        [[s["x2"], s["y2"]] for s in segmentos],
        dtype=np.float64,
    )

    vetores = p2 - p1
    normas = np.linalg.norm(vetores, axis=1)
    unidades = vetores / np.maximum(normas[:, None], 1e-9)

    angulos = (
        np.degrees(np.arctan2(vetores[:, 1], vetores[:, 0]))
        % 180.0
    )
    meios = (p1 + p2) / 2.0

    limite = cfg["merging_global"]
    max_angulo = float(limite["angulo_max_graus"])
    max_distancia = float(
        limite["distancia_perpendicular_max_px"]
    )
    max_gap = float(limite["gap_axial_max_px"])

    def avaliar_blocos(indices_a, indices_b, mesmo_bucket):
        ia_total = np.asarray(indices_a, dtype=np.int64)
        ib = np.asarray(indices_b, dtype=np.int64)

        if ia_total.size == 0 or ib.size == 0:
            return

        bloco_a = 128

        for inicio in range(0, len(ia_total), bloco_a):
            ia = ia_total[inicio:inicio + bloco_a]

            diferencas = (
                np.abs(
                    angulos[ia, None]
                    - angulos[ib][None, :]
                )
                % 180.0
            )

            mascara = (
                np.minimum(diferencas, 180.0 - diferencas)
                <= max_angulo
            )

            if mesmo_bucket:
                mascara &= ib[None, :] > ia[:, None]

            pos_a, pos_b = np.nonzero(mascara)

            if pos_a.size == 0:
                continue

            ai = ia[pos_a]
            bi = ib[pos_b]

            va = unidades[ai]
            vb = unidades[bi].copy()

            inverter = (
                np.einsum("ij,ij->i", va, vb) < 0
            )
            vb[inverter] *= -1.0

            eixos = va + vb
            eixos /= np.maximum(
                np.linalg.norm(eixos, axis=1)[:, None],
                1e-9,
            )

            normais = np.column_stack(
                (-eixos[:, 1], eixos[:, 0])
            )

            distancia = np.abs(
                np.einsum(
                    "ij,ij->i",
                    meios[bi] - meios[ai],
                    normais,
                )
            )

            manter = distancia <= max_distancia

            if not np.any(manter):
                continue

            ai = ai[manter]
            bi = bi[manter]
            eixos = eixos[manter]

            a1 = np.einsum("ij,ij->i", p1[ai], eixos)
            a2 = np.einsum("ij,ij->i", p2[ai], eixos)
            b1 = np.einsum("ij,ij->i", p1[bi], eixos)
            b2 = np.einsum("ij,ij->i", p2[bi], eixos)

            amin = np.minimum(a1, a2)
            amax = np.maximum(a1, a2)
            bmin = np.minimum(b1, b2)
            bmax = np.maximum(b1, b2)

            gap = np.maximum(
                0.0,
                np.maximum(amin, bmin)
                - np.minimum(amax, bmax),
            )

            compativeis = gap <= max_gap

            for i, j in zip(
                ai[compativeis].tolist(),
                bi[compativeis].tolist(),
            ):
                unir(i, j)

    buckets = {}

    for i, segmento in enumerate(segmentos):
        chave = (
            int(segmento["tile_x"]),
            int(segmento["tile_y"]),
        )
        buckets.setdefault(chave, []).append(i)

    pares_processados = set()

    for tx, ty in buckets:
        chave_a = (tx, ty)

        for ny in range(ty - 1, ty + 2):
            for nx in range(tx - 1, tx + 2):
                chave_b = (nx, ny)

                if chave_b not in buckets:
                    continue

                par = (
                    (chave_a, chave_b)
                    if chave_a <= chave_b
                    else (chave_b, chave_a)
                )

                if par in pares_processados:
                    continue

                pares_processados.add(par)

                avaliar_blocos(
                    buckets[par[0]],
                    buckets[par[1]],
                    par[0] == par[1],
                )

    grupos = {}

    for i in range(n):
        grupos.setdefault(raiz(i), []).append(i)

    return [
        agregar_componente(segmentos, indices)
        for indices in grupos.values()
    ]


# Teste de equivalência determinístico.
rng_teste = np.random.default_rng(20260830)
segmentos_teste = []

for _ in range(180):
    tx = int(rng_teste.integers(0, 4))
    ty = int(rng_teste.integers(0, 4))
    x = float(tx * 384 + rng_teste.uniform(0, 512))
    y = float(ty * 384 + rng_teste.uniform(0, 512))
    angulo = float(rng_teste.uniform(0, np.pi))
    comprimento = float(rng_teste.uniform(128, 500))

    segmentos_teste.append({
        "x1": x,
        "y1": y,
        "x2": x + comprimento * np.cos(angulo),
        "y2": y + comprimento * np.sin(angulo),
        "peso": comprimento,
        "confianca": 0.0,
        "comprimento": comprimento,
        "tile_x": tx,
        "tile_y": ty,
    })


def canonico_segmentos(lista):
    return sorted(
        (
            round(s["x1"], 8),
            round(s["y1"], 8),
            round(s["x2"], 8),
            round(s["y2"], 8),
            round(s["comprimento"], 8),
            int(s["fragmentos"]),
        )
        for s in lista
    )


referencia = unir_segmentos_escalar_validacao(
    segmentos_teste
)
vetorizado = unir_segmentos(segmentos_teste)

assert (
    canonico_segmentos(referencia)
    == canonico_segmentos(vetorizado)
), "A implementação vetorizada divergiu da referência."


# Benchmark na amostra Hough já produzida.
inicio = time.perf_counter()
amostra_unida = unir_segmentos(segmentos_amostra)
tempo_amostra = time.perf_counter() - inicio

print(json.dumps({
    "status": "MERGING_VETORIZADO_APROVADO",
    "equivalencia_escalar_vetorizada": True,
    "segmentos_amostra": len(segmentos_amostra),
    "segmentos_apos_merging": len(amostra_unida),
    "tempo_amostra_s": round(tempo_amostra, 4),
    "event_id_preservado": event_id,
    "sha256_config_evento": trava["sha256_config_evento"],
}, indent=2, ensure_ascii=False))

{
  "status": "MERGING_VETORIZADO_APROVADO",
  "equivalencia_escalar_vetorizada": true,
  "segmentos_amostra": 4385,
  "segmentos_apos_merging": 270,
  "tempo_amostra_s": 1.031,
  "event_id_preservado": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "sha256_config_evento": "81ab0148b765fd3664b185812560826b21bf680081aa6c17d37ee2b94f773137"
}


In [10]:
EXECUTAR_EVENTO_ASTA=True
RETOMAR_EVENTO_INCOMPLETO=True

if 'relatorio_smoke' not in globals() or relatorio_smoke.get('status')!='SMOKE_IMPLEMENTACAO_EVENTO_ASTA_APROVADO_SEM_DADOS_ASTA':
    raise RuntimeError('Execute e aprove primeiro o smoke operacional sem dados ASTA.')

def checkpoint_valido(caminho,metodo,id_asta):
    if not caminho.is_file():return None
    registro=json.loads(caminho.read_text(encoding='utf-8'))
    assert verificar_hash_canonico(registro,'sha256_checkpoint')==registro['sha256_checkpoint']
    assert registro['event_id']==EVENT_ID and registro['metodo']==metodo and registro['id_asta']==id_asta
    assert registro['sha256_config_evento']==CONFIG_EVENTO['sha256_config_evento']
    return registro

def detectar_yolo_tiles(modelo,imagem_norm,preenchimento,id_asta):
    segmentos=[];metas=[];lote=[];tempo=0.0;tempo_tiles=0.0
    coordenadas=[(ix,iy,x,y) for iy,y in enumerate(STARTS_Y) for ix,x in enumerate(STARTS_X)]
    barra=tqdm(total=len(coordenadas),desc=f'{id_asta} YOLO tiles',unit='tile',leave=False)
    for inicio in range(0,len(coordenadas),BATCH_TILES_YOLO):
        t_tiles=time.perf_counter()
        bloco=coordenadas[inicio:inicio+BATCH_TILES_YOLO];lote=[];metas=[]
        for ix,iy,x,y in bloco:
            tile=extrair_tile(imagem_norm,x,y,preenchimento)
            lote.append(cv2.cvtColor(tile,cv2.COLOR_GRAY2BGR));metas.append((ix,iy,x,y))
        tempo_tiles+=time.perf_counter()-t_tiles
        torch.cuda.synchronize();t=time.perf_counter()
        resultados=modelo.predict(source=lote,imgsz=cfg['modelo_yolo']['imgsz'],
            conf=cfg['modelo_yolo']['threshold_confianca'],iou=cfg['modelo_yolo']['iou_nms'],
            max_det=cfg['modelo_yolo']['max_det'],device=0,verbose=False,batch=len(lote),half=False)
        assert len(resultados)==len(metas)
        for resultado,(ix,iy,x,y) in zip(resultados,metas):
            if resultado.obb is None:continue
            caixas=resultado.obb.xyxyxyxy.detach().cpu().numpy()
            confs=resultado.obb.conf.detach().cpu().numpy()
            for caixa,confianca in zip(caixas,confs):
                p1,p2=caixa_para_eixo_maior(caixa);p1+=np.array([x,y]);p2+=np.array([x,y])
                centro=(p1+p2)/2
                if not (0<=centro[0]<10560 and 0<=centro[1]<10560):continue
                comprimento=float(np.linalg.norm(p2-p1))
                segmentos.append({'x1':float(p1[0]),'y1':float(p1[1]),'x2':float(p2[0]),'y2':float(p2[1]),
                    'peso':float(confianca),'confianca':float(confianca),'comprimento':comprimento,
                    'tile_x':ix,'tile_y':iy})
        torch.cuda.synchronize();tempo+=time.perf_counter()-t
        barra.update(len(bloco));del resultados,lote
    barra.close();return segmentos,tempo,tempo_tiles

def detectar_hough_tiles(imagem_norm,preenchimento,id_asta):
    d=hough_config['detector'];segmentos=[];tempo=0.0;tempo_tiles=0.0
    coordenadas=[(ix,iy,x,y) for iy,y in enumerate(STARTS_Y) for ix,x in enumerate(STARTS_X)]
    for ix,iy,x,y in tqdm(coordenadas,desc=f'{id_asta} Hough tiles',unit='tile',leave=False):
        t_tiles=time.perf_counter();tile=extrair_tile(imagem_norm,x,y,preenchimento)
        tempo_tiles+=time.perf_counter()-t_tiles;t=time.perf_counter()
        base=tile
        if float(d.get('pre_blur_sigma',0.0))>0:
            base=cv2.GaussianBlur(tile,(0,0),float(d['pre_blur_sigma']))
        bordas=cv2.Canny(base,int(d['canny_lo']),int(d['canny_hi']))
        seed_hough=int(hashlib.sha256(f'{id_asta}|{ix}|{iy}|hough_v3'.encode()).hexdigest()[:8],16)&0x7fffffff
        cv2.setRNGSeed(seed_hough)
        linhas=cv2.HoughLinesP(bordas,1,np.pi/180,threshold=int(d['hough_threshold']),
            minLineLength=max(1,int(round(TILE*float(d['min_length_frac'])))),maxLineGap=int(d['max_gap']))
        if linhas is not None:
            for x1,y1,x2,y2 in np.asarray(linhas).reshape(-1,4):
                p1=np.array([float(x1+x),float(y1+y)]);p2=np.array([float(x2+x),float(y2+y)])
                centro=(p1+p2)/2
                if not (0<=centro[0]<10560 and 0<=centro[1]<10560):continue
                comprimento=float(np.linalg.norm(p2-p1))
                segmentos.append({'x1':float(p1[0]),'y1':float(p1[1]),'x2':float(p2[0]),'y2':float(p2[1]),
                    'peso':comprimento,'confianca':0.0,'comprimento':comprimento,'tile_x':ix,'tile_y':iy})
        tempo+=time.perf_counter()-t
    return segmentos,tempo,tempo_tiles

def criar_registro_frame(metodo,linha,segmentos_brutos,segmentos_unidos,metricas,tempos,meta_norm):
    registro={'protocolo':'asta_external_frame_checkpoint_v3','event_id':EVENT_ID,
        'sha256_config_evento':CONFIG_EVENTO['sha256_config_evento'],'metodo':metodo,
        'id_asta':linha.id_asta,'sha256_imagem':linha.sha256_imagem,'sha256_mascara':linha.sha256_mascara,
        'gt_positivo':bool(int(linha.mascara_pixels_positivos)>0),'segmentos_brutos':len(segmentos_brutos),
        'segmentos_apos_merging':len(segmentos_unidos),'segmentos':segmentos_unidos,
        'metricas':metricas,'normalizacao':meta_norm,'tempos_segundos':tempos,'concluido':True}
    registro['sha256_checkpoint']=sha256_canonico(registro);return registro

def carregar_registros(event_dir,metodo):
    registros=[]
    for caminho in sorted((event_dir/'checkpoints'/metodo).glob('*.json')):
        registro=json.loads(caminho.read_text(encoding='utf-8'))
        assert verificar_hash_canonico(registro,'sha256_checkpoint')==registro['sha256_checkpoint']
        assert registro['event_id']==EVENT_ID and registro['metodo']==metodo
        assert registro['sha256_config_evento']==CONFIG_EVENTO['sha256_config_evento']
        registros.append(registro)
    return registros

def consolidar_metodo(registros,metodo):
    positivos=[r for r in registros if r['gt_positivo']];negativos=[r for r in registros if not r['gt_positivo']]
    pred=sum(r['metricas']['pred_pixels'] for r in positivos);pred_hits=sum(r['metricas']['pred_hits'] for r in positivos)
    gt=sum(r['metricas']['gt_pixels'] for r in positivos);gt_hits=sum(r['metricas']['gt_hits'] for r in positivos)
    p=pred_hits/pred if pred else 0.0;r=gt_hits/gt if gt else 0.0;f1=2*p*r/(p+r) if p+r else 0.0
    macro=float(np.mean([x['metricas']['f1_centerline'] for x in positivos]))
    area_mp=(10560*10560)/1e6
    fp_mp=float(np.mean([x['metricas']['pred_pixels']/area_mp for x in negativos])) if negativos else None
    taxa_fp=float(np.mean([x['metricas']['pred_pixels']>0 for x in negativos])) if negativos else None
    return {'metodo':metodo,'frames':len(registros),'frames_positivos':len(positivos),'frames_negativos':len(negativos),
        'f1_centerline_macro_frames_positivos':macro,'precisao_centerline_pooled':float(p),
        'recall_centerline_pooled':float(r),'f1_centerline_pooled':float(f1),
        'comprimento_fp_por_megapixel_negativo':fp_mp,'taxa_frames_negativos_com_fp':taxa_fp,
        'tempo_preprocessamento_medio_s':float(np.mean([x['tempos_segundos']['preprocessamento'] for x in registros])),
        'tempo_inferencia_medio_s':float(np.mean([x['tempos_segundos']['inferencia'] for x in registros])),
        'tempo_merging_medio_s':float(np.mean([x['tempos_segundos']['merging'] for x in registros])),
        'tempo_total_frame_medio_s':float(np.mean([x['tempos_segundos']['total_frame'] for x in registros]))}

if not EXECUTAR_EVENTO_ASTA:
    print('Evento externo bloqueado. Compartilhe primeiro ASTA_PRONTO_PARA_EVENTO_EXTERNO_UNICO.')
else:
    existente=json.loads(LOCK_PATH.read_text(encoding='utf-8')) if LOCK_PATH.is_file() else None
    if existente and existente.get('status')=='concluido':
        raise RuntimeError('Evento ASTA já concluído. Nova inferência é proibida.')
    if existente:
        if not RETOMAR_EVENTO_INCOMPLETO:raise RuntimeError('Evento incompleto existe. Use RETOMAR_EVENTO_INCOMPLETO=True.')
        assert existente['sha256_contrato_protocolo_asta']==EXPECTED_PROTOCOL_SHA256
        assert existente['sha256_config_evento']==CONFIG_EVENTO['sha256_config_evento']
        EVENT_ID=existente['event_id'];evento=existente
    else:
        if RETOMAR_EVENTO_INCOMPLETO:raise RuntimeError('Não existe evento para retomar.')
        EVENT_ID=str(uuid.uuid4())
        evento={'protocolo':'asta_external_protected_event_v3','status':'em_andamento','event_id':EVENT_ID,
            'iniciado_em_utc':datetime.now(timezone.utc).isoformat(),
            'sha256_contrato_protocolo_asta':EXPECTED_PROTOCOL_SHA256,
            'sha256_config_evento':CONFIG_EVENTO['sha256_config_evento'],
            'frames_previstos':178,'yolo_frames_concluidos':0,'hough_frames_concluidos':0}
        LOCK_PATH.parent.mkdir(parents=True,exist_ok=True)
        with LOCK_PATH.open('x',encoding='utf-8') as arq:json.dump(evento,arq,indent=2,ensure_ascii=False)

    EVENT_DIR=INFERENCE_ROOT/f'evento_{EVENT_ID}'
    for pasta in [EVENT_DIR/'checkpoints/yolo',EVENT_DIR/'checkpoints/hough',EVENT_DIR/'consolidado']:
        pasta.mkdir(parents=True,exist_ok=True)
    config_remota=EVENT_DIR/'config_evento_asta_v3.json'
    if config_remota.exists():
        config_existente=json.loads(config_remota.read_text(encoding='utf-8'))
        assert config_existente['sha256_config_evento']==CONFIG_EVENTO['sha256_config_evento']
    else:gravar_json_atomico(config_remota,CONFIG_EVENTO)

    def atualizar_lock(**campos):
        evento.update(campos);evento['atualizado_em_utc']=datetime.now(timezone.utc).isoformat()
        gravar_json_atomico(LOCK_PATH,evento)

    pendentes_yolo=[]
    for linha in manifest.itertuples():
        if not (EVENT_DIR/'checkpoints/yolo'/f'{linha.id_asta}.json').is_file():pendentes_yolo.append(linha.id_asta)
    modelo=None
    if pendentes_yolo:
        print(f'[{agora()}] Copiando e verificando checkpoint operacional YOLO...')
        copiar_com_hash(peso_operacional,LOCAL_WEIGHT,EXPECTED_OPERATIONAL_WEIGHT_SHA256)
        from ultralytics import YOLO
        modelo=YOLO(str(LOCAL_WEIGHT))

    try:
        for linha in tqdm(manifest.itertuples(),total=len(manifest),desc='Frames ASTA',unit='frame'):
            caminho_yolo=EVENT_DIR/'checkpoints/yolo'/f'{linha.id_asta}.json'
            caminho_hough=EVENT_DIR/'checkpoints/hough'/f'{linha.id_asta}.json'
            reg_yolo=checkpoint_valido(caminho_yolo,'yolo',linha.id_asta) if caminho_yolo.is_file() else None
            reg_hough=checkpoint_valido(caminho_hough,'hough',linha.id_asta) if caminho_hough.is_file() else None
            if reg_yolo is not None and reg_hough is not None:continue

            img_local=STAGING/linha.arquivo_imagem;mask_local=STAGING/linha.arquivo_mascara
            t_copia=time.perf_counter()
            copiar_com_hash(ASTA_RAW/linha.arquivo_imagem,img_local,linha.sha256_imagem)
            copiar_com_hash(ASTA_RAW/linha.arquivo_mascara,mask_local,linha.sha256_mascara)
            tempo_copia=time.perf_counter()-t_copia
            t_pre=time.perf_counter()
            imagem=cv2.imread(str(img_local),cv2.IMREAD_GRAYSCALE)
            if imagem is None:raise RuntimeError(f'Decode da imagem falhou: {linha.id_asta}')
            if imagem.shape!=(10560,10560):raise RuntimeError(f'Dimensão da imagem divergente: {linha.id_asta}')
            imagem_norm,meta_norm=normalizar_frame_uint8(imagem)
            tempo_pre=time.perf_counter()-t_pre
            mascara=cv2.imread(str(mask_local),cv2.IMREAD_GRAYSCALE)
            if mascara is None:raise RuntimeError(f'Decode da máscara falhou: {linha.id_asta}')
            if mascara.shape!=(10560,10560):raise RuntimeError(f'Dimensão da máscara divergente: {linha.id_asta}')
            skeleton_gt=skeletonize(mascara>0)
            preenchimento=int(meta_norm['mediana_normalizada'])

            if reg_yolo is None:
                brutos,t_inf,t_tiles=detectar_yolo_tiles(modelo,imagem_norm,preenchimento,linha.id_asta)
                t_merge=time.perf_counter();unidos=unir_segmentos(brutos);t_merge=time.perf_counter()-t_merge
                t_metric=time.perf_counter();metricas=metricas_centerline(unidos,skeleton_gt,14);t_metric=time.perf_counter()-t_metric
                tempos={'copia_drive':tempo_copia,'preprocessamento':tempo_pre+t_tiles,'inferencia':t_inf,
                    'merging':t_merge,'avaliacao':t_metric,'total_frame':tempo_pre+t_tiles+t_inf+t_merge}
                reg_yolo=criar_registro_frame('yolo',linha,brutos,unidos,metricas,tempos,meta_norm)
                gravar_json_atomico(caminho_yolo,reg_yolo)
                del brutos,unidos

            if reg_hough is None:
                brutos,t_inf,t_tiles=detectar_hough_tiles(imagem_norm,preenchimento,linha.id_asta)
                t_merge=time.perf_counter();unidos=unir_segmentos(brutos);t_merge=time.perf_counter()-t_merge
                t_metric=time.perf_counter();metricas=metricas_centerline(unidos,skeleton_gt,14);t_metric=time.perf_counter()-t_metric
                tempos={'copia_drive':tempo_copia,'preprocessamento':tempo_pre+t_tiles,'inferencia':t_inf,
                    'merging':t_merge,'avaliacao':t_metric,'total_frame':tempo_pre+t_tiles+t_inf+t_merge}
                reg_hough=criar_registro_frame('hough',linha,brutos,unidos,metricas,tempos,meta_norm)
                gravar_json_atomico(caminho_hough,reg_hough)
                del brutos,unidos

            atualizar_lock(yolo_frames_concluidos=len(list((EVENT_DIR/'checkpoints/yolo').glob('*.json'))),
                hough_frames_concluidos=len(list((EVENT_DIR/'checkpoints/hough').glob('*.json'))),
                ultimo_id_concluido=linha.id_asta)
            for caminho in [img_local,mask_local]:
                if caminho.exists():caminho.unlink()
            del imagem,mascara,imagem_norm,skeleton_gt;gc.collect();torch.cuda.empty_cache()

        registros_yolo=carregar_registros(EVENT_DIR,'yolo');registros_hough=carregar_registros(EVENT_DIR,'hough')
        assert len(registros_yolo)==len(registros_hough)==178
        assert {r['id_asta'] for r in registros_yolo}==set(manifest.id_asta.astype(str))
        assert {r['id_asta'] for r in registros_hough}==set(manifest.id_asta.astype(str))

        linhas=[]
        for metodo,registros in [('yolo',registros_yolo),('hough',registros_hough)]:
            for r in registros:
                m=r['metricas'];t=r['tempos_segundos']
                linhas.append({'id_asta':r['id_asta'],'metodo':metodo,'gt_positivo':r['gt_positivo'],
                    'precisao_centerline':m['precisao_centerline'],'recall_centerline':m['recall_centerline'],
                    'f1_centerline':m['f1_centerline'],'pred_pixels':m['pred_pixels'],'gt_pixels':m['gt_pixels'],
                    'pred_hits':m['pred_hits'],'gt_hits':m['gt_hits'],'segmentos_brutos':r['segmentos_brutos'],
                    'segmentos_apos_merging':r['segmentos_apos_merging'],'tempo_preprocessamento':t['preprocessamento'],
                    'tempo_inferencia':t['inferencia'],'tempo_merging':t['merging'],'tempo_total_frame':t['total_frame']})
        tabela=pd.DataFrame(linhas);tabela_path=EVENT_DIR/'consolidado/metricas_por_frame_metodo_asta_v3.csv'
        gravar_csv_atomico(tabela_path,tabela)
        resumo_yolo=consolidar_metodo(registros_yolo,'YOLO11n-OBB seed 20260824')
        resumo_hough=consolidar_metodo(registros_hough,'Hough congelado')

        positivos=tabela[tabela.gt_positivo].pivot(index='id_asta',columns='metodo',values='f1_centerline').dropna()
        assert len(positivos)==177 and {'yolo','hough'}<=set(positivos.columns)
        rng=np.random.default_rng(CONFIG_EVENTO['bootstrap_seed']);boot=[];n=len(positivos)
        valores_y=positivos.yolo.to_numpy(float);valores_h=positivos.hough.to_numpy(float)
        for _ in tqdm(range(CONFIG_EVENTO['bootstrap_replicacoes']),desc='Bootstrap ASTA por frame',unit='rep'):
            idx=rng.integers(0,n,n);my=float(valores_y[idx].mean());mh=float(valores_h[idx].mean())
            boot.append({'f1_yolo':my,'f1_hough':mh,'delta_f1_yolo_menos_hough':my-mh})
        boot_df=pd.DataFrame(boot);boot_path=EVENT_DIR/'consolidado/bootstrap_frames_asta_v3.csv';gravar_csv_atomico(boot_path,boot_df)
        intervalos={c:[float(x) for x in np.percentile(boot_df[c],[2.5,97.5])] for c in boot_df.columns}

        pares=tabela.pivot(index='id_asta',columns='metodo',values='f1_centerline').reset_index()
        pares['delta_yolo_menos_hough']=pares.yolo-pares.hough
        candidatos=pd.concat([pares.nsmallest(12,'yolo'),pares.nsmallest(12,'hough'),
            pares.nsmallest(8,'delta_yolo_menos_hough'),pares.nlargest(8,'delta_yolo_menos_hough')])
        candidatos=candidatos.drop_duplicates('id_asta').sort_values('id_asta')
        candidatos_path=EVENT_DIR/'consolidado/casos_revisao_qualitativa_asta_v3.csv';gravar_csv_atomico(candidatos_path,candidatos)

        resumo={'protocolo':'asta_external_protected_event_v3','status':'EVENTO_EXTERNO_ASTA_CONCLUIDO',
            'event_id':EVENT_ID,'sha256_contrato_protocolo_asta':EXPECTED_PROTOCOL_SHA256,
            'sha256_config_evento':CONFIG_EVENTO['sha256_config_evento'],'frames':178,'positivos':177,'negativos':1,
            'resultado_primario_yolo':resumo_yolo,'hough_congelado':resumo_hough,
            'bootstrap_pareado_frames_positivos':{'replicacoes':2000,'seed':20260823,'n_frames':177,'ic95_percentil':intervalos},
            'aviso_negativos':'ASTA contém somente um frame oficialmente negativo; métricas de falso positivo não têm IC estável.',
            'selecionou_modelo_seed_ou_threshold_com_asta':False,'recalibrou_hough_com_asta':False}
        resumo['sha256_resultado_final']=sha256_canonico(resumo)
        resumo_path=EVENT_DIR/'consolidado/resumo_final_evento_asta_v3.json';gravar_json_atomico(resumo_path,resumo)
        artefatos={p.name:sha256_arquivo(p) for p in [tabela_path,boot_path,candidatos_path,resumo_path]}
        hashes_path=EVENT_DIR/'consolidado/hashes_resultados_asta_v3.json';gravar_json_atomico(hashes_path,artefatos)
        atualizar_lock(status='concluido',concluido_em_utc=datetime.now(timezone.utc).isoformat(),
            yolo_frames_concluidos=178,hough_frames_concluidos=178,
            sha256_resultado_final=resumo['sha256_resultado_final'],
            resumo_relativo=str(resumo_path.relative_to(BACKUP_DIR)).replace('\\','/'))
        print(json.dumps(resumo,indent=2,ensure_ascii=False))
        print('EVENTO EXTERNO ASTA CONCLUÍDO E TRAVADO. Não reexecute nem recalibre.')
    except Exception as exc:
        atualizar_lock(status='incompleto',ultimo_erro=repr(exc))
        raise

[15:52:56] Copiando e verificando checkpoint operacional YOLO...


Frames ASTA:   0%|          | 0/178 [00:00<?, ?frame/s]

ML1_20170902_022840_red Hough tiles:   0%|          | 0/784 [00:00<?, ?tile/s]

ML1_20171023_002210_red YOLO tiles:   0%|          | 0/784 [00:00<?, ?tile/s]

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

...

ML1_20230227_020410_red YOLO tiles:   0%|          | 0/784 [00:00<?, ?tile/s]

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

ML1_20230227_020410_red Hough tiles:   0%|          | 0/784 [00:00<?, ?tile/s]

Bootstrap ASTA por frame:   0%|          | 0/2000 [00:00<?, ?rep/s]

{
  "protocolo": "asta_external_protected_event_v3",
  "status": "EVENTO_EXTERNO_ASTA_CONCLUIDO",
  "event_id": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "sha256_contrato_protocolo_asta": "dc4c58381b1e3e7d2a24f7fdac8fc59276e53b9d900853c1191a497319e862bc",
  "sha256_config_evento": "81ab0148b765fd3664b185812560826b21bf680081aa6c17d37ee2b94f773137",
  "frames": 178,
  "positivos": 177,
  "negativos": 1,
  "resultado_primario_yolo": {
    "metodo": "YOLO11n-OBB seed 20260824",
    "frames": 178,
    "frames_positivos": 177,
    "frames_negativos": 1,
    "f1_centerline_macro_frames_positivos": 0.5833671994488407,
    "precisao_centerline_pooled": 0.8669670628233774,
    "recall_centerline_pooled": 0.5362403417154702,
    "f1_centerline_pooled": 0.6626286499353983,
    "comprimento_fp_por_megapixel_negativo": 10.402318640955006,
    "taxa_frames_negativos_com_fp": 1.0,
    "tempo_preprocessamento_medio_s": 3.141777901972068,
    "tempo_inferencia_medio_s": 3.753354924921346,
    "tempo_mer

## 5. Verificação pós-evento
Execute após a conclusão. Esta célula não carrega modelos nem realiza inferência; apenas
valida a trava, os 356 checkpoints e os artefatos consolidados no Drive.

In [11]:
if not LOCK_PATH.is_file():
    print('Evento externo ainda não foi iniciado.')
else:
    trava=json.loads(LOCK_PATH.read_text(encoding='utf-8'))
    if trava.get('status')!='concluido':
        print(json.dumps({'status':'EVENTO_ASTA_INCOMPLETO','event_id':trava.get('event_id'),
            'yolo_frames_concluidos':trava.get('yolo_frames_concluidos',0),
            'hough_frames_concluidos':trava.get('hough_frames_concluidos',0)},indent=2,ensure_ascii=False))
    else:
        event_dir=INFERENCE_ROOT/f"evento_{trava['event_id']}";consolidado=event_dir/'consolidado'
        resumo=json.loads((consolidado/'resumo_final_evento_asta_v3.json').read_text(encoding='utf-8'))
        assert verificar_hash_canonico(resumo,'sha256_resultado_final')==trava['sha256_resultado_final']
        hashes=json.loads((consolidado/'hashes_resultados_asta_v3.json').read_text(encoding='utf-8'))
        for nome,esperado in tqdm(sorted(hashes.items()),desc='Verificando resultados ASTA',unit='arq'):
            assert sha256_arquivo(consolidado/nome)==esperado
        for metodo in ['yolo','hough']:
            arquivos=sorted((event_dir/'checkpoints'/metodo).glob('*.json'));assert len(arquivos)==178
            ids=[]
            for caminho in tqdm(arquivos,desc=f'Verificando checkpoints {metodo}',unit='frame'):
                r=json.loads(caminho.read_text(encoding='utf-8'))
                assert verificar_hash_canonico(r,'sha256_checkpoint')==r['sha256_checkpoint']
                assert r['event_id']==trava['event_id'] and r['metodo']==metodo
                assert r['sha256_config_evento']==trava['sha256_config_evento'];ids.append(r['id_asta'])
            assert len(set(ids))==178 and set(ids)==set(manifest.id_asta.astype(str))
        print(json.dumps({'status':'BACKUP_EVENTO_EXTERNO_ASTA_V3_APROVADO',
            'event_id':trava['event_id'],'sha256_contrato_protocolo_asta':EXPECTED_PROTOCOL_SHA256,
            'sha256_config_evento':trava['sha256_config_evento'],
            'sha256_resultado_final':trava['sha256_resultado_final'],
            'yolo_frames':178,'hough_frames':178,'reexecucao_proibida':True},indent=2,ensure_ascii=False))

Verificando resultados ASTA:   0%|          | 0/4 [00:00<?, ?arq/s]

Verificando checkpoints yolo:   0%|          | 0/178 [00:00<?, ?frame/s]

Verificando checkpoints hough:   0%|          | 0/178 [00:00<?, ?frame/s]

{
  "status": "BACKUP_EVENTO_EXTERNO_ASTA_V3_APROVADO",
  "event_id": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "sha256_contrato_protocolo_asta": "dc4c58381b1e3e7d2a24f7fdac8fc59276e53b9d900853c1191a497319e862bc",
  "sha256_config_evento": "81ab0148b765fd3664b185812560826b21bf680081aa6c17d37ee2b94f773137",
  "sha256_resultado_final": "eef0f468754b195984a4888cc8c8439f12b5e9db3af94e875082272503542f7c",
  "yolo_frames": 178,
  "hough_frames": 178,
  "reexecucao_proibida": true
}
